## RAG Basics
A simple RAG pipeline has five parts: documents, chunking, embeddings, vector storage, and retrieval-augmented answering. 
You split documents into small chunks, convert each chunk into an embedding vector, store those vectors in a vector store, 
then at query time you embed the user question, find similar chunks, and send those chunks to the model as extra context.

Why this matters: the model does not need to memorize your data, and your answers become grounded in your own files. 
This is especially useful for internal docs, FAQs, notes, API docs, resumes, or company policies.

How it fits your pattern
In your current pattern, you already know Agent, model, tools, and MCP servers. 
RAG fits into that same mental model by acting like a knowledge retrieval tool that the agent can use before responding.

So instead of only giving your agent browser and filesystem MCP servers, you can also give it a retrieval capability. Conceptually, your flow becomes:

User asks a question.

Retriever finds matching chunks from your local knowledge base.

Agent receives those chunks as context.

Agent answers using the retrieved content.

# In-Memory RAG Example

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from agents.mcp import MCPServerStdio
import os
from pathlib import Path
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel

In [2]:
load_dotenv(override=True)

True

In [3]:
groq_api_key = os.getenv('GROQ_API_KEY')
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
groq_model = OpenAIChatCompletionsModel(model="meta-llama/llama-4-scout-17b-16e-instruct", openai_client=groq_client)

#### Creation of Knolweldge base

mkdir -p sandbox/knowledge

echo '# NATO\nNorth Atlantic Treaty Organization, founded 1949' > sandbox/knowledge/nato.md

In [4]:
sandbox_path = Path(os.getcwd()) / "sandbox"
knowledge_path = sandbox_path / "knowledge"
output_path = sandbox_path / "output"

paths = [sandbox_path, knowledge_path, output_path]
paths

[PosixPath('/Users/himanshuramranjan/Documents/AI/agents/6_mcp/community_contributions/mcp_with_rag/sandbox'),
 PosixPath('/Users/himanshuramranjan/Documents/AI/agents/6_mcp/community_contributions/mcp_with_rag/sandbox/knowledge'),
 PosixPath('/Users/himanshuramranjan/Documents/AI/agents/6_mcp/community_contributions/mcp_with_rag/sandbox/output')]

In [54]:
def load_markdown_documents(folder: Path):
    docs = []
    for file_path in folder.glob("*.md"):
        text = file_path.read_text(encoding="utf-8")
        docs.append({
            "filename": file_path.name,
            "content": text
        })
    return docs

In [56]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [57]:
def build_chunks():
    docs = load_markdown_documents(knowledge_path)
    all_chunks = []

    for doc in docs:
        chunks = chunk_text(doc["content"])
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "filename": doc["filename"],
                "chunk_id": idx,
                "text": chunk
            })
    return all_chunks

In [75]:
@function_tool
def search_knowledge_base(query: str) -> str:
    """
    Search the local markdown knowledge base and return the most relevant chunks.

    Args:
        query: The user question to search for in the knowledge base.
    """
    chunks = build_chunks()
    query_words = query.lower().split()

    scored = []
    for chunk in chunks:
        text_lower = chunk["text"].lower()
        score = sum(1 for word in query_words if word in text_lower)
        if score > 0:
            scored.append((score, chunk))

    scored.sort(reverse=True, key=lambda x: x[0])
    top_chunks = scored[:3]

    if not top_chunks:
        return "No relevant content found in the knowledge base."

    results = []
    for score, chunk in top_chunks:
        results.append(
            f"Source: {chunk['filename']} | Chunk: {chunk['chunk_id']} | Score: {score}\n"
            f"{chunk['text']}"
        )

    return "\n\n---\n\n".join(results)

In [59]:
@function_tool
def write_output_file(filename: str, content: str) -> str:
    """
    Write content to a markdown file inside sandbox/output.

    Args:
        filename: Name of the markdown file to create.
        content: Content to write into the file.
    """
    if not filename.endswith(".md"):
        filename += ".md"

    file_path = output_path / filename
    file_path.write_text(content, encoding="utf-8")
    return f"Written successfully to {file_path}"

In [73]:
files_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", str(sandbox_path)]
}

In [81]:
instructions = """
You are a strict RAG agent.

Rules:
1. For every question, ALWAYS call search_knowledge_base first.
2. ONLY answer from the retrieved content.
3. If nothing relevant is found, say: "This information is not available in the knowledge base."
4. After preparing the final answer, you MUST write that final answer to the requested markdown file using the filesystem MCP server.
5. Even if the answer is "not available in the knowledge base", you MUST still write that exact message to the file.
6. Never skip the file-writing step when the user asks for a file.
"""

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=30) as mcp_server_files:
    agent = Agent(
        name="rag-agent",
        instructions=instructions,
        model=groq_model,
        tools=[search_knowledge_base],
        mcp_servers=[mcp_server_files]
    )

    with trace("rag-phase-2-mcp-write"):
        result = await Runner.run(
            agent,
            "How many moons does Jupiter have?. Write the final answer, even if not found, to answer.md inside the sandbox folder."
        )

# RAG implemenation using Vector DB locally (ChromaDB)

#### Installation steps for ChromaDB

python -m venv .venv
source .venv/bin/activate  # Mac/Linux

-- .venv\Scripts\activate  # Windows

python -m ensurepip --upgrade
python -m pip install --upgrade pip

python -m pip install chromadb sentence-transformers numpy pathlib

#### Creation of Knolweldge base

mkdir -p sandbox/knowledge

echo '# NATO\nNorth Atlantic Treaty Organization, founded 1949' > sandbox/knowledge/nato.md

In [106]:
import chromadb  # The vector database (like SQLite but for AI embeddings)
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction # Chroma's wrapper for free, local embeddings

In [99]:
db_path = Path("./sandbox/chroma_db")
db_path.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(db_path))

In [ ]:
# convert sentences to 384 numbers (embedding) using a local LLM Model (90 MB in size)
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2",
    device="cpu",
    normalize_embeddings=False
)

In [101]:
 # Creates/opens a table called knowledge_base with built-in embeddings, similar liek : CREATE TABLE knowledge_base (id, vector, metadata)
collection = client.get_or_create_collection(
    name="knowledge_base",
    embedding_function=embedding_fn
)

In [102]:
# 3) Index your files (one-time operation)
def index_knowledge():
    knowledge_path = Path("./sandbox/knowledge")  # Your .md files
    docs = []    # Text chunks
    metadatas = []  # {"filename": "nato.md", "chunk_id": 0}
    ids = []     # "doc_0_chunk_0", "doc_0_chunk_1"
    
    for i, file_path in enumerate(knowledge_path.glob("*.md")):
        content = file_path.read_text()           # Read NATO.md
        chunks = chunk_text(content)              # Split into 500-char pieces 
        
        for j, chunk in enumerate(chunks):
            docs.append(chunk)                    # "NATO = North Atlantic..."
            metadatas.append({"filename": file_path.name, "chunk_id": j})
            ids.append(f"doc_{i}_chunk_{j}")      # Unique ID per chunk
    
    collection.add(documents=docs, metadatas=metadatas, ids=ids)
    print(f"Indexed {len(docs)} chunks")

index_knowledge()

Indexed 1 chunks


In [104]:
# 4) Retrieval tool
@function_tool
def search_embed_knowledge_base(query: str) -> str:
    """Semantic search over the local knowledge base using ChromaDB embeddings."""
    results = collection.query(
        query_texts=[query],              # "What does NATO stand for?"
        n_results=3,                      # Top 3 matches
        include=["documents", "metadatas", "distances"]
    )
    
    output = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        output.append(f"📄 {meta['filename']} | Score: {1-dist:.3f}\n```\n{doc}\n```")
    
    return "\n---\n\n".join(output) or "No relevant content found."

In [105]:
instructions = """
You are a strict RAG agent.

Rules:
1. For every question, ALWAYS call search_embed_knowledge_base first.
2. ONLY answer from the retrieved content.
3. If nothing relevant is found, say: "This information is not available in the knowledge base."
4. After preparing the final answer, you MUST write that final answer to the requested markdown file using the filesystem MCP server.
5. Even if the answer is "not available in the knowledge base", you MUST still write that exact message to the file.
6. Never skip the file-writing step when the user asks for a file.
"""

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=30) as mcp_server_files:
    agent = Agent(
        name="rag-agent",
        instructions=instructions,
        model=groq_model,
        tools=[search_embed_knowledge_base],
        mcp_servers=[mcp_server_files]
    )

    with trace("rag-phase-2-mcp-write"):
        result = await Runner.run(
            agent,
            "When was NATO founded ?. Write the final answer, even if not found, to output/answer.md."
        )

### Flow for the RAG :

User: "When was NATO founded?"

↓ Agent calls tool ↓

search_embed_knowledge_base("When was NATO founded?")

↓ Chroma returns ↓

"📄 nato.md | Score: 0.95\n1949 founding..."

↓ Agent answers ↓

"NATO was founded in 1949" ✓


Conversion of sentences into a mapped numbers :

```
"NATO was founded in 1949" 
    ↓ AI magic ↓
[0.12, -0.34, 0.89, ..., 0.21]  # 384 numbers

"What is NATO?" 
    ↓ AI magic ↓
[0.11, -0.33, 0.91, ..., 0.20]  # Very similar numbers!
```